Implement GPT model from scratch for text generation

In [129]:
import torch
import torch.nn as nn

In [130]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

Dummy GPT Model: Use a placeholder for layer norm and transformer block

In [ ]:
#GPT Model

# Where:
# nn.Sequential is a container module that allows you to stack these layers together (sequentially) to form a complete model
# nn.Linear is a linear transformation layer that applies a linear transformation to incoming data
# linear transformation is the process of mapping input features to output features using a linear equation



class DummyGPTModel(nn.Module):

    def __init__(self, cfg):

        # initialize the model with config params
        super().__init__()

        # embedding layers for token + positions
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])

        # dropout layer
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        #transformer blocks
        self.trf_blocks = nn.Sequential(*[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        # final normalization step
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])

        # output head for logits, where logits are the unnormalized predictions for each token in the vocabulary
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    #forward pass is the process of passing input data through the model to obtain predictions/outputs
    def forward(self, in_idx):

        # in_idx is input tensor and this line unpacks the shape of the input tensor into two variables: batch_size and seq_len
        batch_size, seq_len = in_idx.shape

        # gives the token embeddings for the input indices, where each index corresponds to a token in the vocabulary
        tok_embeds = self.tok_emb(in_idx)

        # gives the positional embeddings for the sequence length, where each position corresponds to a token's position in the input sequence
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))

        # combines token + positional embeddings, applies dropout, passes through transformer blocks, and normalizes the output
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)

        # computes the logits (unnormalized predictions) for each token in the vocabulary based on the final output of the model
        logits = self.out_head(x)

        return logits


class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):
        return x


class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()

    def forward(self, x):
        return x

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

text1 = "Every effort moves you"
text2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(text1)))

batch.append(torch.tensor(tokenizer.encode(text2)))

batch = torch.stack(batch)

print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [ ]:
torch.manual_seed(42)

batch_example = torch.randn(2,5)

layer =nn.Sequential(nn.Linear(5,6), nn.ReLU())

out = layer(batch_example)

print(out)

tensor([[0.0000, 0.1842, 0.0052, 0.7233, 0.0000, 0.5298],
        [0.0000, 0.0000, 0.0000, 0.2237, 0.0000, 0.7727]],
       grad_fn=<ReluBackward0>)


In [135]:
mean = out.mean(dim = -1, keepdim=True)

variance = out.var(dim = -1, keepdim = True)

out_norm = (out - mean) / torch.sqrt(variance)

print("Normalized layer: ", out_norm)

print("Mean:", mean)

print("Variance:", variance)

Normalized layer:  tensor([[-0.7672, -0.1794, -0.7506,  1.5410, -0.7672,  0.9234],
        [-0.5351, -0.5351, -0.5351,  0.1857, -0.5351,  1.9546]],
       grad_fn=<DivBackward0>)
Mean: tensor([[0.2404],
        [0.1661]], grad_fn=<MeanBackward1>)
Variance: tensor([[0.0982],
        [0.0963]], grad_fn=<VarBackward0>)
